# 07 Local LLM

This notebook builds the prompt bundle from TF-IDF and topic outputs, then optionally runs a local model.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, *list(cwd.parents[:3])]
PROJECT_ROOT = next((path for path in candidates if (path / "working" / "lib" / "pipeline.py").exists()), cwd)
LIB_ROOT = PROJECT_ROOT / "working" / "lib"
if str(LIB_ROOT) not in sys.path:
    sys.path.insert(0, str(LIB_ROOT))

import pipeline as xp

xp.ensure_working_tree()
print("Project root:", PROJECT_ROOT)
display(xp.list_subjects())


## Local Model Options

Default path: **Ollama + `isotnek/qwen3.5:9B-Unsloth-UD-Q4_K_XL`**

Why this is the default:

- easiest for most people to install
- available across macOS, Linux, and Windows
- stronger model quality while still staying practical for local use

Optional upgrade:

- `qwen2.5:14b-instruct` if you want to stay on a smaller official Ollama model family


In [ ]:
display(xp.local_llm_options())
print("Detected backends:", xp.available_local_llm_backends())


In [ ]:
SUBJECT = "costco"
DATASETS = ["domain", "codomain"]
TOP_TERMS = 12
TOP_TOPICS = None
BACKEND = "ollama"
MODEL = "isotnek/qwen3.5:9B-Unsloth-UD-Q4_K_XL"
RUN_LLM = True
MAX_TOKENS = 3072
AUTO_INSTALL_OLLAMA = False


## Ollama Setup

This cell detects the OS, checks whether Ollama is already installed, and only attempts the matching install path if `AUTO_INSTALL_OLLAMA = True`.


In [ ]:
import shutil
import subprocess

system = xp.detect_os()
print("Detected OS:", system)
print("Ollama installed:", xp.ollama_installed())
print("Model storage:", xp.ollama_model_storage_hint()["model_dir"])

if not xp.ollama_installed():
    if not AUTO_INSTALL_OLLAMA:
        print("Ollama is not installed. Set AUTO_INSTALL_OLLAMA=True to let this cell install it automatically.")
    else:
        if system == "darwin":
            cmd = ["brew", "install", "ollama"]
        elif system == "linux":
            cmd = ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"]
        elif system == "windows":
            cmd = ["winget", "install", "--id", "Ollama.Ollama"]
        else:
            raise RuntimeError(f"Unsupported OS for automated install: {system}")

        print("Running install command:", " ".join(cmd))
        subprocess.run(cmd, check=True)

print("Ollama installed after setup:", xp.ollama_installed())


In [ ]:
server_ok = xp.start_ollama_server(timeout_seconds=20)
print("Ollama server running:", server_ok)


In [ ]:
model_status = xp.ensure_ollama_model(MODEL)
model_status


In [ ]:
if xp.ollama_installed() and xp.ollama_server_running():
    smoke = xp.run_local_llm("Reply with READY only.", backend="ollama", model=MODEL, max_tokens=32)
    print(smoke)
else:
    print("Skipping smoke test because Ollama is not ready.")


In [ ]:
subject = xp.get_subject_config(SUBJECT)
bundles = {}
bundle_rows = []
for dataset in DATASETS:
    bundles[dataset] = xp.prepare_local_llm_stage(
        subject,
        dataset,
        top_terms=TOP_TERMS,
        top_topics=TOP_TOPICS,
    )
    bundle_rows.append(
        {
            "dataset": dataset,
            "prompt_path": bundles[dataset]["prompt_path"],
            "deterministic_output_path": bundles[dataset]["deterministic_output_path"],
        }
    )
display(pd.DataFrame(bundle_rows))


In [ ]:
ollama_ready = xp.ollama_installed() and xp.ollama_server_running() if BACKEND == "ollama" else True

results = []
for dataset in DATASETS:
    if RUN_LLM and ollama_ready:
        result = xp.run_local_llm_stage(
            subject,
            dataset,
            backend=BACKEND,
            model=MODEL,
            max_tokens=MAX_TOKENS,
        )
    elif RUN_LLM and BACKEND == "ollama" and not ollama_ready:
        result = {
            "dataset": dataset,
            "backend": BACKEND,
            "model": MODEL,
            "note": "Skipping LLM run because Ollama is not installed or the server is not running.",
            "prompt_path": bundles[dataset]["prompt_path"],
        }
    else:
        result = {
            "dataset": dataset,
            "backend": BACKEND,
            "model": MODEL,
            "note": "Prompt bundle created. Set RUN_LLM=True to execute a local backend.",
            "prompt_path": bundles[dataset]["prompt_path"],
        }
    results.append(result)

display(
    pd.DataFrame(
        [
            {
                key: value
                for key, value in result.items()
                if key != "parsed_output"
            }
            for result in results
        ]
    )
)


In [ ]:
import json
from pathlib import Path

for result in results:
    print(f"\nLLM output: {result.get('dataset', 'unknown')}")
    if result.get("parsed_output") is not None:
        display(result["parsed_output"])
    elif result.get("result_path"):
        output_path = Path(result["result_path"])
        print(output_path.read_text(encoding="utf-8")[:4000])
    else:
        print("No LLM output was produced. If that was intentional, set RUN_LLM=False. Otherwise check the setup cells above.")


## Optional Shutdown

You do not have to stop the Ollama server after the notebook finishes, but you can if you want to free memory.


In [ ]:
# Optional
# xp.stop_ollama_server()


In [ ]:
preview_subject = subject if "subject" in globals() else xp.get_subject_config(SUBJECT)
_ = xp.show_stage_figures(preview_subject, "07_local_llm")
